In [ ]:
#| hide
from varga.core import *
from varga.schema import *

# varga

> what kind of thing a document is, and the fields it holds

Two questions about one document. `guess_type` answers the first from cue phrases and
countable evidence, with no model unless two kinds tie. `as_schema` and `structured` answer the
second against a shape you name or invent.

## Install

``` sh
uv add varga            # doctypes; no model, no network
uv add 'varga[llm]'     # ...and `structured`, which asks one
```

## What is it

In [ ]:
INVOICE = '''# INVOICE
Invoice No: ACM-2024-0117
Date: 2024-03-01
Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd
Total due: 1,240.00 EUR
Payment terms: net 30
'''
r = guess_type(INVOICE)
r.doctype, r.decisive, round(r.margin, 2)

('invoice', True, 0.4)

`decisive` is the whole point: it says whether the cues settled it. When two kinds tie,
`margin` is 0 and a caller knows to spend a model call. Nothing else here costs one.

In [ ]:
signals(INVOICE).counts

{'money': 1, 'date': 1, 'ref': 1, 'heading': 1, 'org': 2}

## What is in it

In [ ]:
from dataclasses import fields
[f.name for f in fields(as_schema('invoice'))]

['number',
 'date',
 'due_date',
 'vendor',
 'vendor_tax_id',
 'bill_to',
 'ship_to',
 'currency',
 'subtotal',
 'tax',
 'total',
 'payment_terms',
 'items']

Ten shapes are built in: `invoice`, `purchase_order`, `quote`, `receipt`, `catalogue`,
`contract`, `resume`, `paper`, `meeting_notes`, `other`. Or write one on the spot:

In [ ]:
[(f.name, f.type.__name__) for f in fields(as_schema('vendor:str, total:float, items:list'))]

[('vendor', 'str'), ('total', 'float'), ('items', 'list')]

`structured(chat, prompt, schema)` fills it. It prefers the model's constrained mode and
falls back to a JSON reply when that raises, so a model without tool calling still answers.

## The doctypes

In [ ]:
len(DOCTYPES), sorted(DOCTYPES)

Ten of them are work-product labels: `proposal`, `presentation`, `requirements_spec`,
`technical_design`, `regulatory_guidance`, `procedure`, `qa_artifact`, `roadmap`, `claim`,
`clinical_record`, and each is decisive on its own text without a model. Prose that merely
borrows the vocabulary scores under 0.2 against every one of them.

## What it depends on

`fastcore` and [rahasya](https://github.com/vedicreader/rahasya), for the honorific-anchored
name regex the entity leg shares with the privacy gate. One definition of what a name is.

`rishi` is needed only by `structured`. `vruksha` adds keyphrases to `signals().ents` and is
display only: no doctype score reads them, and `signals` works without it.

## Development

The notebooks in `nbs/` are the source; the modules are generated.

``` sh
pip install -e .
nbdev_prepare
```